# Script 1: Cohort Construction
This notebook constructs the ARF NIV study cohort from CLIF tables.

## Steps
1. Construct hospital blocks.
2. Identify ED encounters.
3. Apply inclusion and exclusion criteria.

#### 1. Set up and constructing hospital blocks

Load configuration and clinical thresholds, register source CLIF tables, and group eligible hospitalizations into hospital blocks using the configured time-gap rule.

Create block-level summaries and a key linking each hospitalization to its hospital block for downstream joins.

In [0]:
# --- Load required dependenies ---
import yaml
import pathlib
import pandas as pd
from datetime import datetime
from time import perf_counter

# Count elapsed time
notebook_start = perf_counter()

# --- Load global and local configuration ---
# Project root
project_root = pathlib.Path.cwd()

# Load configuration files
while not (project_root / "config").is_dir():
  if project_root.parent == project_root:
    raise Exception("Configuration folder not found")
  project_root = project_root.parent

with open(project_root / "config" / "config.yaml", encoding = "utf-8") as f:
  config = yaml.safe_load(f)

with open(project_root / "config" / "global_config.yaml", encoding = "utf-8") as f:
  global_config = yaml.safe_load(f)

# Assign global config values
config_sql = {key: value for key, value in global_config.items() if key != "required_tables"}

# Assign local config values
tables_location = config["tables_location"]
project_location = config["project_location"]
institution = config["institution"]
time_zone = config["time_zone"]
file_type = config["file_type"]

# --- Loading project related clinical thresholds ---
while not (project_root / "outlier-thresholds").is_dir():
  if project_root.parent == project_root:
    raise Exception("Threshold folder not found")
  project_root = project_root.parent

thresholds_path = project_root / "outlier-thresholds" / "project_outlier_thresholds.csv"
thresholds = pd.read_csv(thresholds_path).set_index("variable_name")

# --- Expose configuration and/or thresholds to SQL as needed ---
config_sql["age_min"] = float(thresholds.loc["age_at_admission", "lower_limit"])
config_sql["age_max"] = float(thresholds.loc["age_at_admission", "upper_limit"])
config_sql["ph_cutoff"] = float(thresholds.loc["ph_cutoff", "lower_limit"])
config_sql["sf_cutoff"] = float(thresholds.loc["sf_cutoff", "lower_limit"])
config_sql["spo2_eligibility"] = float(thresholds.loc["spo2_eligibility", "upper_limit"])
config_sql["ph_arterial_min"] = float(thresholds.loc["ph_arterial", "lower_limit"])
config_sql["ph_arterial_max"] = float(thresholds.loc["ph_arterial", "upper_limit"])
config_sql["ph_venous_min"] = float(thresholds.loc["ph_venous", "lower_limit"])
config_sql["ph_venous_max"] = float(thresholds.loc["ph_venous", "upper_limit"])
config_sql["spo2_min"] = float(thresholds.loc["spo2", "lower_limit"])
config_sql["spo2_max"] = float(thresholds.loc["spo2", "upper_limit"])
config_sql["fio2_min"] = float(thresholds.loc["fio2_set", "lower_limit"])
config_sql["fio2_max"] = float(thresholds.loc["fio2_set", "upper_limit"])

spark.createDataFrame([config_sql]).createOrReplaceTempView("config_sql")

# --- Load CLIF tables ---
# Check if tables exist
clif_table_filenames = {item.name.removesuffix("/") for item in dbutils.fs.ls(tables_location) if item.isDir()}

required_tables = set(global_config["required_tables"])

missing_table = sorted(required_tables - clif_table_filenames)

if len(missing_table) > 0:
  raise Exception(f"Missing tables: {missing_table}")
else: 
  print("All tables present.")

# Register source CLIF tables as temp views
for table_name in required_tables:
  source = spark.read.parquet(f"{tables_location}/{table_name}")
  source.createOrReplaceTempView(f"src_{table_name}")
  print(f"Registered temp view for {table_name}")


In [0]:
%sql
-- Create column that stores the time of previous discharge
-- Create column that stores the gap in hours between current admission and prior discharge
-- Flag a hospitalization if it represents a new hospital block
-- Compute cumulative block number per patient (e.g., first hospital block =1 , second = 2, etc.)
-- Determine end of hospitalization time for discharge location
create or replace temp view working_hospitalization as
with step1 as (
  select
    h.*,
    c.hospital_block_time_hrs as hospital_block_time_hrs,
    lag(h.discharge_dttm, 1) over (
        partition by h.patient_id
        order by h.admission_dttm
      ) as prev_discharge_dttm
  from
    src_clif_hospitalization as h cross join config_sql as c
  where
    h.age_at_admission >= c.age_min
    and h.admission_dttm >= cast(c.study_start as timestamp)
),
step2 as (
  select
    *,
    case
      when
        prev_discharge_dttm is null
        or cast(admission_dttm as timestamp) - cast(prev_discharge_dttm as timestamp)
          > hospital_block_time_hrs * interval '1' hour
      then
        1
      else 0
    end as new_block
  from
    step1
)
select
  * except (hospital_block_time_hrs, prev_discharge_dttm, new_block),
  sum(coalesce(new_block, 1)) over (
      partition by patient_id
      order by admission_dttm
      rows between unbounded preceding and current row
    ) as block_num,
  coalesce(discharge_dttm, admission_dttm) as end_ts
from
  step2
order by
  patient_id,
  admission_dttm;

In [0]:
%sql
-- Stores information on grouped hospitalizations (hospital blocks)
-- Stores the patient age on first admission within block
-- Stores the start of the hospital block time
-- Stores the end of the hospital block time
-- Stores the discharge location from last admission within block
-- Stores the number of hospitalizations within block
create or replace temp view hospital_blocks as
with step1 as (
  select
    *,
    row_number() over (
        partition by patient_id, block_num
        order by end_ts desc NULLS last, hospitalization_id asc
      ) as index_by_descending_end_ts
  from
    working_hospitalization
)
select
  s.patient_id,
  s.block_num,
  min(s.age_at_admission) as age_at_admission,
  min(s.admission_dttm) as block_start_admit,
  max(s.discharge_dttm) as block_end_discharge,
  count(*) as num_hospitalizations,
  max(
    case
      when s.index_by_descending_end_ts = 1 then s.discharge_category
      else NULL
    end
  ) as discharge_location,
  concat(s.patient_id, '_', cast(min(s.admission_dttm) as string)) as hospital_block_id
from
  step1 as s cross join config_sql as c
group by
  s.patient_id,
  s.block_num
having
  min(s.admission_dttm) <= min(c.study_end)
  and min(s.age_at_admission) <= min(c.age_max);

In [0]:
%sql
-- Create key for each hospitalization_id to link to hospitalization block
create or replace temp view hospital_block_key_obj as
select
  w.patient_id,
  w.hospitalization_id,
  h.hospital_block_id,
  h.block_start_admit,
  h.block_end_discharge
from
  working_hospitalization w
    inner join hospital_blocks h
      on w.patient_id = h.patient_id
      and w.block_num = h.block_num;

In [0]:
%sql
-- Joining hospital_block_key_obj to source CLIF table: CLIF hospitalization
create or replace temp view clif_hospitalization as
select
  s.*,
  h.hospital_block_id,
  h.block_start_admit,
  h.block_end_discharge
from
  src_clif_hospitalization s
    inner join hospital_block_key_obj h
      on s.hospitalization_id = h.hospitalization_id
      and s.patient_id = h.patient_id;

In [0]:
%sql
create temp table hospital_block_key_obj_tt as
select * from hospital_block_key_obj;

In [0]:
%sql
-- Joining hospital_block_key_obj to source CLIF table: CLIF ADT
create or replace temp view clif_adt as
select
  s.hospitalization_id,
  s.hospital_id,
  s.hospital_type,
  s.in_dttm,
  s.out_dttm,
  s.location_name,
  s.location_category,
  s.location_type,
  h.patient_id,
  h.hospital_block_id,
  h.block_start_admit,
  h.block_end_discharge
from
  src_clif_adt s
    inner join hospital_block_key_obj_tt h
      on s.hospitalization_id = h.hospitalization_id;

In [0]:
%sql
-- Joining hospital_block_key_obj to source CLIF table: CLIF respiratory support
create or replace temp view clif_respiratory_support as
select
    s.*,
    h.patient_id,
    h.hospital_block_id,
    h.block_start_admit,
    h.block_end_discharge
from src_clif_respiratory_support s
inner join hospital_block_key_obj_tt h
    on s.hospitalization_id = h.hospitalization_id;

In [0]:
%sql
-- Joining hospital_block_key_obj to source CLIF table: CLIF continuous meds
create or replace temp view clif_medication_admin_continuous as
select 
    s.*,
    h.patient_id,
    h.hospital_block_id,
    h.block_start_admit,
    h.block_end_discharge
from src_clif_medication_admin_continuous s
cross join config_sql c
inner join hospital_block_key_obj_tt h
    on s.hospitalization_id = h.hospitalization_id
where s.admin_dttm >= cast(c.study_start as timestamp)
    and s.med_group = "vasoactives";

In [0]:
%sql
-- Joining hospital_block_key_obj to source CLIF table: CLIF labs
create or replace temp view clif_labs as
select 
    s.* except(hospitalization_id),
    cast(s.hospitalization_id as string) as hospitalization_id,
    h.patient_id,
    h.hospital_block_id,
    h.block_start_admit,
    h.block_end_discharge
from src_clif_labs s
inner join hospital_block_key_obj_tt h
    on cast(s.hospitalization_id as string) = h.hospitalization_id
where cast(s.lab_category as string) in ("ph_arterial", "ph_venous");

In [0]:
%sql 
-- Joining hospital_block_key_obj to source CLIF table: CLIF vitals
create or replace temp view clif_vitals as
select s.* except(hospitalization_id),
    cast(s.hospitalization_id as string) as hospitalization_id,
    h.patient_id,
    h.hospital_block_id,
    h.block_start_admit,
    h.block_end_discharge
from src_clif_vitals s
inner join hospital_block_key_obj_tt h
    on cast(s.hospitalization_id as string) = h.hospitalization_id
where s.vital_category = 'spo2';

In [0]:
%sql
-- Joining hospital_block_key_obj to source CLIF table: CLIF CRRT
create or replace temp view clif_crrt_therapy as
select
    cast(s.hospitalization_id as string) as hospitalization_id,
    s.crrt_mode_category,
    s.recorded_dttm,
    h.patient_id,
    h.hospital_block_id,
    h.block_start_admit,
    h.block_end_discharge
from src_clif_crrt_therapy s
inner join hospital_block_key_obj_tt h
    on cast(s.hospitalization_id as string) = h.hospitalization_id;

#### 2. Group ED blocks and select encounters that start in the ED

In [0]:
%sql
-- Create a final cohort that tracks ADT events associated with eligible block IDs
create or replace temp view final_cohort as
select 
    a.*,
    lower(a.location_category) as location_lower,
    array_contains(c.ed_allowed_locations, lower(a.location_category)) ed_equivalent
from clif_adt a
cross join config_sql c
where exists(select *
             from clif_hospitalization h
             where h.hospital_block_id = a.hospital_block_id)
order by a.hospital_block_id, a.in_dttm;
  

In [0]:
%sql
-- Identify and select only blocks that start in the ED 
create or replace temp view final_cohort_1 as
with step1 as (
    select
        *,
        first_value(location_lower) over block_window as first_location,
        first_value(hospital_id) over block_window as first_hospital_id,
        first_value(hospital_type) over block_window as first_hospital_type,
        last_value(hospital_id) over block_window as last_hospital_id,
        last_value(hospital_type) over block_window as last_hospital_type
    from final_cohort
    window block_window as (
        partition by hospital_block_id
        order by in_dttm
        rows between unbounded preceding and unbounded following
    )
)
select * except (first_location)
from step1
where first_location = 'ed';

In [0]:
%sql
-- Identify and select only rows before triage to ICU, stepdown, ward, etc. 
create or replace temp view final_cohort_2 as
with numbered as (
    select
        *,
        row_number() over (
            partition by hospital_block_id
            order by in_dttm
        ) as row_num
    from final_cohort_1
),
indexed as (
    select
        *,
        min(
            case when ed_equivalent = false then row_num end
        ) over (
            partition by hospital_block_id
        ) as triage_idx
    from numbered
),
labeled as (
    select
        *,
        case
            when triage_idx is null then 'discharge'
            else max(
                case when row_num = triage_idx then location_category end
            ) over (
                partition by hospital_block_id
            )
        end as triage_location,
        case
            when triage_idx is null then 'discharge'
            else max(
                case when row_num = triage_idx then location_name end
            ) over (
                partition by hospital_block_id
            )
        end as triage_location_raw
    from indexed
),
before_triage as (
    select *
    from labeled
    where triage_idx is null
       or row_num < triage_idx
),
with_ed_cutoff as (
    select
        *,
        max(
            case when location_lower = 'ed' then out_dttm end
        ) over (
            partition by hospital_block_id
        ) as max_ed_out
    from before_triage
)
select *
from with_ed_cutoff
where out_dttm <= max_ed_out;

In [0]:
%sql
-- Summarize final cohort information
create or replace temp view final_cohort_3 as
select 
    hospital_block_id,
    min(patient_id) as patient_id,
    min(in_dttm) as start_ed,
    max(out_dttm) as stop_ed,
    timestampdiff(minute, min(in_dttm), max(out_dttm))
    / 60.0 as ed_hours,
    min(triage_location) as triage_location,
    min(triage_location_raw) as triage_location_raw,
    min(first_hospital_id) as first_hospital_id,
    min(first_hospital_type) as first_hospital_type,
    min(last_hospital_id) as last_hospital_id,
    min(last_hospital_type) as last_hospital_type
from final_cohort_2
group by hospital_block_id;

In [0]:
%sql
create temp table final_cohort_3_tt as
select * from final_cohort_3;

In [0]:
# --- Tracking cohort counts after each inclusion/exclusion step ---
cohort = pd.DataFrame({
    "step": pd.Series(dtype = "int64"),
    "description": pd.Series(dtype = "string"),
    "n_patients": pd.Series(dtype = "int64"),
    "n_encounters": pd.Series(dtype = "int64")
    })

# Small helper function to acquire cohort counts using Spark SQL
def update_cohort_counts(step, description, view_name):
    global cohort

    sql_query = sql(f"""
        SELECT
            COUNT(DISTINCT patient_id) AS n_patients,
            COUNT(*) AS n_encounters
        FROM {view_name}
        """).first()
    
    new_row = pd.DataFrame({"step": [step], "description": [description], "n_patients": [sql_query.n_patients], "n_encounters": [sql_query.n_encounters]})

    cohort = pd.concat([cohort, new_row], ignore_index=True)
    
# Update cohort count table
update_cohort_counts(step = 0, description = "All adult ED encounters within date range", view_name = "final_cohort_3")

# Show current cohort counts
display(cohort)

#### 3. Filtering on inclusion criteria

1. NIV (NIPPY, high flow NC) starting in the ED
2. Continuous NIV for at least 4 hours after starting

In [0]:
%sql
-- Inclusion criteria 1. NIV started in the ED
create or replace temp view final_cohort_4 as
with step1 as (
  select distinct
    f.*,
    r.recorded_dttm,
    r.device_category
  from
    final_cohort_3_tt f
      cross join config_sql c
      inner join clif_respiratory_support r
        on f.hospital_block_id = r.hospital_block_id
            and r.recorded_dttm between f.start_ed and f.stop_ed
  where array_contains(c.niv_devices, lower(r.device_category))
),
step2 as (
  select
    *,
    row_number() over (partition by hospital_block_id order by recorded_dttm) as niv_idx
  from
    step1
),
step3 as (
  select
    *
  from
    step2
  where
    niv_idx = 1
)
select
  * except (device_category, niv_idx, recorded_dttm),
  recorded_dttm as niv_start,
  device_category as first_niv_device,
  recorded_dttm + INTERVAL '4' hour as t_0
from
  step3;

In [0]:
# Update cohort count table
update_cohort_counts(step = 1, description = "Inclusion 1: Where NIV is started in the ED", view_name = "final_cohort_4")

# Show current cohort counts
display(cohort)

In [0]:
%sql
-- Inclusion criteria 2. continuous NIV for at least 4 hours after starting
create or replace temp view final_cohort_5 as
with step1 as (
    select distinct 
        f.*,
        r.recorded_dttm,
        r.device_category
    from final_cohort_4 f
    cross join config_sql c
    inner join clif_respiratory_support r
        on f.hospital_block_id = r.hospital_block_id
            and (r.recorded_dttm between f.niv_start and f.t_0)
            and r.device_category is not NULL 
            and not array_contains(c.niv_devices, lower(r.device_category))
)
select 
    *
from final_cohort_4 f
where not exists (
    select *
    from step1 s1
    where s1.hospital_block_id = f.hospital_block_id
)


In [0]:
# Update cohort count table
update_cohort_counts(step = 2, description = "Inclusion 2: Continuous NIV for at least 4 hours after starting", view_name = "final_cohort_5")

# Show current cohort counts
display(cohort)

#### 4. Exclusion criteria
1. Leave ED (admitted or discharged) within 4 hours after t0 (defined as 4 hours after starting NIV)
2. Receive IMV before t0 (i.e., within 4 hours after starting NIV)
3. Receive vasopressors before t0
4. Receive CRRT before t0
5. Most recent pH <= 7.1 at t0
6. Most recent SF <= 120 at t0
7. Code status set to "comfort care only" at t0

In [0]:
%sql
-- Exclusion criterion 1: retain blocks whose ED departure is strictly after the cutoff at t0
create or replace temp view final_cohort_6 as
select
    *
from final_cohort_5
where stop_ed > t_0;

In [0]:
# Update cohort count table
update_cohort_counts(step = 3, description = "Exclusion 1: Leave ED (admitted or discharged) within 4 hours after t0", view_name = "final_cohort_6")

# Show current cohort counts
display(cohort)

In [0]:
%sql
-- Exclusion criterion 2: retain blocks whose first IMV or tracheostomy is strictly after the cutoff at t0
create or replace temp view final_cohort_7 as
with step1 as (
    select
        f.hospital_block_id,
        r.recorded_dttm as first_imv_dttm,
        row_number() over (
            partition by f.hospital_block_id
            order by r.recorded_dttm asc NULLS last
        ) as imv_idx
    from final_cohort_6 f
    inner join (
        select
            hospital_block_id,
            recorded_dttm
        from clif_respiratory_support
        where lower(device_category) = 'imv'
    ) r
        on f.hospital_block_id = r.hospital_block_id
),
imv_events as (
    select
        hospital_block_id,
        first_imv_dttm
    from step1
    where imv_idx = 1
),
step3 as (
    select
        f.hospital_block_id,
        r.recorded_dttm as first_trach_dttm,
        row_number() over (
            partition by f.hospital_block_id
            order by r.recorded_dttm asc NULLS last
        ) as trach_idx
    from final_cohort_6 f
    inner join (
        select
            hospital_block_id,
            recorded_dttm
        from clif_respiratory_support
        where tracheostomy = 1
    ) r
        on f.hospital_block_id = r.hospital_block_id
), 
trach_events as (
    select
        hospital_block_id,
        first_trach_dttm
    from step3
    where trach_idx = 1
)
select 
    f.*,
    i.first_imv_dttm,
    t.first_trach_dttm
from final_cohort_6 f
left join imv_events i
    on f.hospital_block_id = i.hospital_block_id
left join trach_events t
    on f.hospital_block_id = t.hospital_block_id
where (i.first_imv_dttm is NULL or i.first_imv_dttm > f.t_0)
    and (t.first_trach_dttm is NULL or t.first_trach_dttm > f.t_0);

In [0]:
%sql
create temp table final_cohort_7_tt as
select * from final_cohort_7;

In [0]:
# Update cohort count table
update_cohort_counts(step = 4, description = "Exclusion 2: IMV (intubation or trach) before t0", view_name = "final_cohort_7_tt")

# Show current cohort counts
display(cohort)

In [0]:
%sql
-- Exclusion criterion 3. Vasopressers received before t0
create or replace temp view final_cohort_8 as
with ranked_pressors as (
  select
    m.hospital_block_id,
    m.admin_dttm as first_vasoactive_time,
    m.med_category as first_vasoactive_name,
    m.med_dose as first_vasoactive_dose,
    m.med_dose_unit as first_vasoactive_unit,
    row_number() over (
        partition by m.hospital_block_id
        order by m.admin_dttm asc nulls last
      ) as pressor_idx
  from
    clif_medication_admin_continuous m
      left semi join final_cohort_7_tt f
        on m.hospital_block_id = f.hospital_block_id
  where
    lower(m.med_group) = 'vasoactives'
    and lower(m.mar_action_name) <> 'stopped'
    and m.med_dose is not null
    and not isnan(m.med_dose)
    and m.med_dose > 0
),
first_pressors as (
  select
    hospital_block_id,
    first_vasoactive_time,
    first_vasoactive_name,
    first_vasoactive_dose,
    first_vasoactive_unit
  from
    ranked_pressors
  where
    pressor_idx = 1
)
select
  f.*,
  p.first_vasoactive_time,
  p.first_vasoactive_name,
  p.first_vasoactive_dose,
  p.first_vasoactive_unit
from
  final_cohort_7_tt f
    left join first_pressors p
      on f.hospital_block_id = p.hospital_block_id
where
  p.first_vasoactive_time is null
  or p.first_vasoactive_time > f.t_0;

In [0]:
# Update cohort count table
update_cohort_counts(step = 5, description = "Exclusion 3: Receive vasopressors before t0", view_name = "final_cohort_8")

# Show current cohort counts
display(cohort)

In [0]:
%sql
-- Exclusion criterion 4: exclude blocks with CRRt at or before t_0
create or replace temp view final_cohort_9 as
with ranked_crrt as (
  select
    r.hospital_block_id,
    r.crrt_mode_category as first_crrt_mode,
    r.recorded_dttm as first_crrt_dttm,
    row_number() over (
        partition by r.hospital_block_id
        order by r.recorded_dttm nulls last
      ) as crrt_idx
  from
    clif_crrt_therapy r
      inner join final_cohort_8 f
        on r.hospital_block_id = f.hospital_block_id
),
first_crrt as (
  select
    hospital_block_id,
    first_crrt_mode,
    first_crrt_dttm
  from
    ranked_crrt
  where
    crrt_idx = 1
)
select
  f.*,
  c.first_crrt_mode,
  c.first_crrt_dttm
from
  final_cohort_8 f
    left join first_crrt c
      on f.hospital_block_id = c.hospital_block_id
where
  c.first_crrt_dttm is null
  or c.first_crrt_dttm > f.t_0;

In [0]:
# Update cohort count table
update_cohort_counts(step = 6, description = "Exclusion 4 : CRRT before t0", view_name = "final_cohort_9")

# Show current cohort counts
display(cohort)

In [0]:
%sql
-- Exclusion criterion 5: exclude blocks whose latest valid ph before t_0 is <= 7.1
create or replace temp view final_cohort_10 as
with ph_values as (
  select /*+ BROADCAST(f) */
    l.hospital_block_id,
    l.lab_result_dttm,
    l.lab_category,
    try_cast(l.lab_value as double) as ph_value
  from
    clif_labs l
      inner join final_cohort_9 f
        on l.hospital_block_id = f.hospital_block_id
        and l.lab_result_dttm < f.t_0
  where
    l.lab_category in ('ph_arterial', 'ph_venous')
),
valid_ph as (
  select
    p.*
  from
    ph_values p cross join config_sql c
  where
    (
      p.lab_category = 'ph_arterial'
      and p.ph_value between c.ph_arterial_min and c.ph_arterial_max
    )
    or (
      p.lab_category = 'ph_venous'
      and p.ph_value between c.ph_venous_min and c.ph_venous_max
    )
),
ranked_ph as (
  select 
    *,
    row_number() over (partition by hospital_block_id order by lab_result_dttm desc) as ph_idx
  from
    valid_ph
),
severe_ph as (
  select 
    p.hospital_block_id
  from
    ranked_ph p cross join config_sql c
  where
    p.ph_idx = 1
    and p.ph_value <= c.ph_cutoff
)
select 
  f.*
from
  final_cohort_9 f
where
  not exists (
    select
      *
    from
      severe_ph p
    where
      p.hospital_block_id = f.hospital_block_id
  );

In [0]:
%sql
create temp table final_cohort_10_tt as
select * from final_cohort_10;

In [0]:
# Update cohort count table
update_cohort_counts(step = 7, description = "Exclusion 5 : Exclude blocks whose latest valid ph before t0 is <= 7.1", view_name = "final_cohort_10_tt")

# Show current cohort counts
display(cohort)

In [0]:
%sql
-- Exclusion criterion 6: Most recent SF <= 120 at t0
-- Step 1: select the latest valid spo2 before t_0
create or replace temp view recent_spo2 as
with spo2_values as (
    select
        v.hospital_block_id,
        v.recorded_dttm as vital_dttm,
        try_cast(v.vital_value as double) as spo2
    from clif_vitals v
    inner join final_cohort_10_tt f
        on v.hospital_block_id = f.hospital_block_id
        and v.recorded_dttm < f.t_0
    where v.vital_category = 'spo2'
),
valid_spo2 as (
    select v.*
    from spo2_values v
    cross join config_sql c
    where v.spo2 between c.spo2_min and c.spo2_max
),
ranked_spo2 as (
    select
        *,
        row_number() over (
            partition by hospital_block_id
            order by vital_dttm desc
        ) as spo2_idx
    from valid_spo2
)
select
    s.hospital_block_id,
    s.vital_dttm,
    s.spo2
from ranked_spo2 s
cross join config_sql c
where s.spo2_idx = 1
    and s.spo2 <= c.spo2_eligibility;

In [0]:
%sql
-- Exclusion criterion 6: Most recent SF <= 120 at t0
-- Step 2: fill fio2 within device runs and calculate SF
create or replace temp view final_cohort_11 as
with respiratory_values as (
    select
        r.hospital_block_id,
        r.recorded_dttm,
        r.device_category,
        nanvl(
            try_cast(r.fio2_set as double),
            cast(null as double)
        ) as fio2_set
    from clif_respiratory_support r
    left semi join final_cohort_10_tt f
        on r.hospital_block_id = f.hospital_block_id
),
valid_respiratory as (
    select r.*
    from respiratory_values r
    cross join config_sql c
    where r.fio2_set is null
        or r.fio2_set between c.fio2_min and c.fio2_max
),
numbered as (
    select
        *,
        row_number() over (
            partition by hospital_block_id
            order by recorded_dttm nulls last
        ) as event_seq
    from valid_respiratory
),
with_previous_device as (
    select
        *,
        lag(device_category) over (
            partition by hospital_block_id
            order by event_seq
        ) as previous_device
    from numbered
),
run_starts as (
    select
        *,
        case
            when event_seq = 1 then 1
            when device_category <=> previous_device then 0
            else 1
        end as new_device_run
    from with_previous_device
),
device_runs as (
    select
        *,
        sum(new_device_run) over (
            partition by hospital_block_id
            order by event_seq
            rows between unbounded preceding and current row
        ) as device_run
    from run_starts
),
forward_filled as (
    select
        *,
        last_value(fio2_set) ignore nulls over (
            partition by hospital_block_id, device_run
            order by event_seq
            rows between unbounded preceding and current row
        ) as fio2_forward
    from device_runs
),
backward_filled as (
    select
        *,
        first_value(fio2_forward) ignore nulls over (
            partition by hospital_block_id, device_run
            order by event_seq
            rows between current row and unbounded following
        ) as fio2_filled
    from forward_filled
),
ranked_fio2 as (
    select
        r.hospital_block_id,
        r.fio2_filled,
        s.spo2,
        row_number() over (
            partition by r.hospital_block_id
            order by r.event_seq desc
        ) as fio2_idx
    from backward_filled r
    inner join recent_spo2 s
        on r.hospital_block_id = s.hospital_block_id
        and r.recorded_dttm < s.vital_dttm
),
sf_values as (
    select
        hospital_block_id,
        try_divide(spo2, fio2_filled) as sf_ratio
    from ranked_fio2
    where fio2_idx = 1
),
severe_sf as (
    select s.hospital_block_id
    from sf_values s
    cross join config_sql c
    where s.sf_ratio is not null
        and s.sf_ratio <= c.sf_cutoff
)
select f.*
from final_cohort_10_tt f
where not exists (
    select 1
    from severe_sf s
    where s.hospital_block_id = f.hospital_block_id
);

In [0]:
# Update cohort count table
update_cohort_counts(step = 8, description = "Exclusion 6: Most recent SF <= 120 at t0", view_name = "final_cohort_11")

# Show current cohort counts
display(cohort)

In [0]:
%sql
-- Exclusion criterion 6: Code status "comfort care only" at t_0
create or replace temp view final_cohort_12 as
with ranked_status as (
    select
        f.hospital_block_id,
        c.code_status_category as code_status_t0,
        row_number() over (
            partition by f.hospital_block_id
            order by c.start_dttm desc
        ) as status_idx
    from final_cohort_11 f
    inner join src_clif_code_status c
        on f.patient_id = c.patient_id
        and c.start_dttm <= f.t_0
),
recent_status as (
    select
        hospital_block_id,
        code_status_t0
    from ranked_status
    where status_idx = 1
)
select
    f.*,
    s.code_status_t0
from final_cohort_11 f
left join recent_status s
    on f.hospital_block_id = s.hospital_block_id
where s.code_status_t0 is null
    or lower(s.code_status_t0) <> 'and';

In [0]:
# Update cohort count table
update_cohort_counts(step = 9, description = "Exclusion 6: Code status \"comfort care only\" at t_0", view_name = "final_cohort_12")

# Show current cohort counts
display(cohort)

In [0]:
%sql
-- Exclusion criterion 6: Hospital is not IMC capable
create or replace temp view final_cohort_14 as
with hospital_counts as (
    select
        first_hospital_id,
        count(*) as total_rows,
        sum(
            case
                when lower(triage_location) = 'stepdown' then 1
                else 0
            end
        ) as imc_rows
    from final_cohort_12
    group by first_hospital_id
),
capable_hospitals as (
    select h.first_hospital_id
    from hospital_counts h
    cross join config_sql c
    where h.imc_rows * 1.0 / h.total_rows >= c.imc_capable_cutoff
)
select f.*
from final_cohort_12 f
inner join capable_hospitals h
    on f.first_hospital_id <=> h.first_hospital_id;

In [0]:
# Update cohort count table
update_cohort_counts(step = 10, description = "Exclusion 7: Encounters at hospitals that are not IMC capable", view_name = "final_cohort_14")

# Show current cohort counts
display(cohort)

In [0]:
%sql
-- Create a cohort from outside COVID period
create or replace temp view no_pandemic_cohort as
select 
    f.*
from final_cohort_14 f
cross join config_sql c
where f.start_ed < cast(c.covid_start as timestamp)
    or f.start_ed > cast(c.covid_end as timestamp);

In [0]:
# Update cohort count table
update_cohort_counts(step = 11, description = "Sensitivity Analysis: Exclude encounters presenting to ED during COVID pandemic", view_name = "no_pandemic_cohort")

# Show current cohort counts
display(cohort)

#### 5. Final cohort materialization
Saving the final cohort to a temporary table that is accessible from within the current Spark session.

In [0]:
%sql
create or replace temp table no_pandemic_one_encounter_per_patient as
-- Randomly select one block per patient
with ranked_encounters as (
    select
        *,
        row_number() over (
            partition by patient_id
            order by
                -- a hash function that creates psuedo-random numbers inside blocks of the same patient
                sha2(
                    concat('1|', cast(hospital_block_id as string)),
                    256
                ),
                hospital_block_id
        ) as patient_encounter_idx
    from no_pandemic_cohort
)
select
    * except (patient_encounter_idx)
from ranked_encounters
where patient_encounter_idx = 1;

In [0]:
print(f"elapsed time for 0_cohort_constrution from start to end: {(perf_counter() - notebook_start)/60:.2f} minutes")

This script creates only the temporary tables needed for processing this and any following ETL scripts. Running it assumes authorized access to the relevant source data, and temporary tables are accessible only within the creating user’s session. No permanent patient-level tables or patient-level export files, including those containing PHI, are created, and source data are not overwritten. Temporary tables become inaccessible when the session ends, with underlying storage cleaned up automatically in the background.